# 07. XGBoost Classifier — Rheumatoid Arthritis Classification

## 1. Project Context and Target Definition

**Task**: Binary retrospective classification of recorded diagnoses.
- `Target_RA = 1`: Rheumatoid Arthritis (≈23.6%)
- `Target_RA = 0`: All six other original diagnosis categories combined into Non-RA (≈76.4%)

**Dataset**: 12,085 records with 14 candidate clinical features.

**Model Owner**: Anjani Kumar (with team contributions)

**Important**: This is a retrospective classification system for recorded diagnoses, not a clinically validated early-detection or prediction system.

---

### Model Development Protocol
- All preprocessing fitted **only** on training data.
- Hyperparameters selected via internal validation split.
- Threshold selected on validation data **before** any test evaluation.
- Test set evaluated **once** after all decisions are frozen.
- `Disease`, `Target_RA`, and `patient_row_id` are **never** model input features.

## 2. Imports and Reproducibility

In [1]:
import sys
import os
import json
import time
import warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for saving plots
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    precision_recall_curve, roc_curve
)

import xgboost as xgb

# Reproducibility seeds
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# Print versions for reproducibility
import sklearn
print(f"Python:       {sys.version}")
print(f"NumPy:        {np.__version__}")
print(f"Pandas:       {pd.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"XGBoost:      {xgb.__version__}")
print(f"Joblib:       {joblib.__version__}")
print(f"Matplotlib:   {matplotlib.__version__}")

Python:       3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]
NumPy:        2.3.0
Pandas:       2.3.0
scikit-learn: 1.9.0
XGBoost:      3.2.0
Joblib:       1.5.3
Matplotlib:   3.10.3


## 3. Load Shared Training and Test Partitions

In [2]:
# Add project root to sys.path
root = Path("..").resolve() if Path("..").resolve().name == "RA_Project" else Path(".").resolve()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.data_loader import (
    load_train_data,
    load_test_data,
    get_feature_names,
    get_feature_metadata,
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
    TARGET_COLUMN,
    ID_COLUMN,
)

# Load reproducible stratified train and test partitions
X_train_full, y_train_full = load_train_data(return_X_y=True)
X_test, y_test = load_test_data(return_X_y=True)

# Load full DataFrames for patient_row_id (traceability)
train_df_full = load_train_data(return_X_y=False)
test_df_full = load_test_data(return_X_y=False)
test_row_ids = test_df_full[ID_COLUMN].values

feature_names = get_feature_names()

print(f"X_train_full shape: {X_train_full.shape}")
print(f"y_train_full shape: {y_train_full.shape}  (RA: {y_train_full.sum()} [{y_train_full.mean():.2%}])")
print(f"X_test shape:       {X_test.shape}")
print(f"y_test shape:       {y_test.shape}  (RA: {y_test.sum()} [{y_test.mean():.2%}])")
print(f"\nStandardized Features ({len(feature_names)}): {feature_names}")
print(f"Numeric features:     {NUMERIC_FEATURES}")
print(f"Categorical features: {CATEGORICAL_FEATURES}")

X_train_full shape: (9668, 14)
y_train_full shape: (9668,)  (RA: 2278 [23.56%])
X_test shape:       (2417, 14)
y_test shape:       (2417,)  (RA: 570 [23.58%])

Standardized Features (14): ['Age', 'Gender', 'ESR', 'CRP', 'RF', 'Anti-CCP', 'HLA-B27', 'ANA', 'Anti-Ro', 'Anti-La', 'Anti-dsDNA', 'Anti-Sm', 'C3', 'C4']
Numeric features:     ['Age', 'ESR', 'CRP', 'RF', 'Anti-CCP', 'C3', 'C4']
Categorical features: ['Gender', 'HLA-B27', 'ANA', 'Anti-Ro', 'Anti-La', 'Anti-dsDNA', 'Anti-Sm']


## 4. Verify Schema, Labels, and Partition Sizes

In [3]:
# Verify dimensions
assert X_train_full.shape == (9668, 14), f"Expected (9668, 14), got {X_train_full.shape}"
assert y_train_full.shape == (9668,), f"Expected (9668,), got {y_train_full.shape}"
assert X_test.shape == (2417, 14), f"Expected (2417, 14), got {X_test.shape}"
assert y_test.shape == (2417,), f"Expected (2417,), got {y_test.shape}"

# Verify no leakage columns
for col in ['Disease', 'Target_RA', 'patient_row_id']:
    assert col not in X_train_full.columns, f"Leakage detected: '{col}' in X_train!"
    assert col not in X_test.columns, f"Leakage detected: '{col}' in X_test!"

# Verify feature order matches expected
assert list(X_train_full.columns) == feature_names, "Feature order mismatch in X_train!"
assert list(X_test.columns) == feature_names, "Feature order mismatch in X_test!"

# Verify binary labels
assert set(y_train_full.unique()).issubset({0, 1}), "y_train must be strictly binary"
assert set(y_test.unique()).issubset({0, 1}), "y_test must be strictly binary"

# Document missingness
print("=== Missing Values in Training Data ===")
missing_train = X_train_full.isnull().sum()
missing_pct = (missing_train / len(X_train_full) * 100).round(2)
missing_df = pd.DataFrame({'Missing': missing_train, 'Pct': missing_pct})
print(missing_df[missing_df['Missing'] > 0].to_string())

print("\n=== Categorical Value Counts (Training) ===")
for col in CATEGORICAL_FEATURES:
    print(f"  {col}: {dict(X_train_full[col].value_counts(dropna=False))}")

print("\n✅ All schema, label, and partition size verifications passed.")

=== Missing Values in Training Data ===
            Missing    Pct
ESR             863   8.93
CRP            1929  19.95
RF             1075  11.12
Anti-CCP       2599  26.88
HLA-B27        1543  15.96
ANA            3022  31.26
Anti-Ro        2303  23.82
Anti-La        2389  24.71
Anti-dsDNA     3768  38.97
Anti-Sm        4192  43.36
C3             1367  14.14
C4             1631  16.87

=== Categorical Value Counts (Training) ===
  Gender: {'Male': np.int64(4927), 'Female': np.int64(4741)}
  HLA-B27: {'Positive': np.int64(4961), 'Negative': np.int64(3164), nan: np.int64(1543)}
  ANA: {'Positive': np.int64(4191), nan: np.int64(3022), 'Negative': np.int64(2455)}
  Anti-Ro: {'Positive': np.int64(4231), 'Negative': np.int64(3134), nan: np.int64(2303)}
  Anti-La: {'Positive': np.int64(4260), 'Negative': np.int64(3019), nan: np.int64(2389)}
  Anti-dsDNA: {nan: np.int64(3768), 'Positive': np.int64(3282), 'Negative': np.int64(2618)}
  Anti-Sm: {nan: np.int64(4192), 'Positive': np.int64(2979)

## 5. Create Internal Training/Validation Split

Split the 9,668-record training partition into:
- Internal training subset (~80%, ~7,734 records)
- Validation subset (~20%, ~1,934 records)

The 2,417-record test set remains **untouched** until all decisions are finalized.

In [4]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_train_full
)

print(f"Internal training: {X_train.shape[0]} records  (RA: {y_train.sum()} [{y_train.mean():.2%}])")
print(f"Validation:        {X_val.shape[0]} records  (RA: {y_val.sum()} [{y_val.mean():.2%}])")
print(f"Held-out test:     {X_test.shape[0]} records  (RA: {y_test.sum()} [{y_test.mean():.2%}])")

assert X_train.shape[0] == 7734, f"Expected 7734 training records, got {X_train.shape[0]}"
assert X_val.shape[0] == 1934, f"Expected 1934 validation records, got {X_val.shape[0]}"
print("\n✅ Internal train/validation split created successfully.")

Internal training: 7734 records  (RA: 1822 [23.56%])
Validation:        1934 records  (RA: 456 [23.58%])
Held-out test:     2417 records  (RA: 570 [23.58%])

✅ Internal train/validation split created successfully.


## 6. Preprocessing and Missing-Value Handling

**Strategy**:
- **Numeric features**: XGBoost natively handles `NaN` values via learned missing-value routing. We pass numeric `NaN` values through without imputation to leverage this.
- **Categorical features**: Standard `XGBClassifier` cannot consume raw strings. We use:
  - `SimpleImputer(strategy='constant', fill_value='Missing')` to make NaN an explicit category.
  - `OneHotEncoder(handle_unknown='ignore', sparse_output=False)` fitted only on training data.
- **No global scaling** is applied (tree-based models are scale-invariant).
- All transformers are fitted on the **internal training subset only**.

In [5]:
# Build preprocessing pipeline
# Numeric: passthrough (preserve NaN for XGBoost native handling)
numeric_transformer = Pipeline(steps=[
    ('passthrough', 'passthrough')
])

# Categorical: impute missing as 'Missing' category, then one-hot encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, NUMERIC_FEATURES),
        ('cat', categorical_transformer, CATEGORICAL_FEATURES)
    ],
    verbose_feature_names_out=True
)

# Fit ONLY on internal training subset
preprocessor.fit(X_train)

# Transform all splits
X_train_processed = preprocessor.transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

# Get transformed feature names
transformed_feature_names = preprocessor.get_feature_names_out().tolist()

print(f"Transformed feature count: {len(transformed_feature_names)}")
print(f"Transformed features: {transformed_feature_names}")
print(f"\nX_train_processed shape: {X_train_processed.shape}")
print(f"X_val_processed shape:   {X_val_processed.shape}")
print(f"X_test_processed shape:  {X_test_processed.shape}")

# Verify numeric NaN values are preserved
num_nans_train = np.isnan(X_train_processed[:, :len(NUMERIC_FEATURES)]).sum()
print(f"\nNaN values preserved in numeric columns (training): {num_nans_train}")
print("✅ Preprocessor fitted on training data only. NaN preserved for XGBoost native handling.")

Transformed feature count: 27
Transformed features: ['num__Age', 'num__ESR', 'num__CRP', 'num__RF', 'num__Anti-CCP', 'num__C3', 'num__C4', 'cat__Gender_Female', 'cat__Gender_Male', 'cat__HLA-B27_Missing', 'cat__HLA-B27_Negative', 'cat__HLA-B27_Positive', 'cat__ANA_Missing', 'cat__ANA_Negative', 'cat__ANA_Positive', 'cat__Anti-Ro_Missing', 'cat__Anti-Ro_Negative', 'cat__Anti-Ro_Positive', 'cat__Anti-La_Missing', 'cat__Anti-La_Negative', 'cat__Anti-La_Positive', 'cat__Anti-dsDNA_Missing', 'cat__Anti-dsDNA_Negative', 'cat__Anti-dsDNA_Positive', 'cat__Anti-Sm_Missing', 'cat__Anti-Sm_Negative', 'cat__Anti-Sm_Positive']

X_train_processed shape: (7734, 27)
X_val_processed shape:   (1934, 27)
X_test_processed shape:  (2417, 27)

NaN values preserved in numeric columns (training): 7627
✅ Preprocessor fitted on training data only. NaN preserved for XGBoost native handling.


## 7. Baseline XGBoost Model

Train a baseline model with sensible defaults, using early stopping on the validation set.

In [6]:
# Class imbalance calculation from training labels only
n_neg_train = int((y_train == 0).sum())
n_pos_train = int((y_train == 1).sum())
scale_pos_weight_value = n_neg_train / n_pos_train
print(f"Training class distribution: Negative={n_neg_train}, Positive={n_pos_train}")
print(f"Calculated scale_pos_weight: {scale_pos_weight_value:.4f}")

# Baseline configuration
baseline_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'n_estimators': 500,
    'learning_rate': 0.05,
    'max_depth': 4,
    'min_child_weight': 3,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.0,
    'reg_lambda': 1.0,
    'random_state': RANDOM_STATE,
    'n_jobs': -1,
    'tree_method': 'hist',
    'early_stopping_rounds': 30,
}

print("\n--- Training Baseline (no class weighting) ---")
baseline_model = xgb.XGBClassifier(**baseline_params)
t0 = time.time()
baseline_model.fit(
    X_train_processed, y_train,
    eval_set=[(X_train_processed, y_train), (X_val_processed, y_val)],
    verbose=False
)
baseline_time = time.time() - t0

baseline_best_iter = baseline_model.best_iteration
print(f"Best iteration: {baseline_best_iter}")
print(f"Training time: {baseline_time:.2f}s")

# Evaluate baseline on validation
val_proba_baseline = baseline_model.predict_proba(X_val_processed)[:, 1]
val_prauc_baseline = average_precision_score(y_val, val_proba_baseline)
val_rocauc_baseline = roc_auc_score(y_val, val_proba_baseline)
val_logloss_baseline = float(baseline_model.evals_result()['validation_1']['logloss'][baseline_best_iter])

# Also compute at default threshold 0.5
val_pred_baseline = (val_proba_baseline >= 0.5).astype(int)
val_f1_baseline = f1_score(y_val, val_pred_baseline)
val_recall_baseline = recall_score(y_val, val_pred_baseline)
val_precision_baseline = precision_score(y_val, val_pred_baseline)

print(f"\nBaseline Validation Metrics (threshold=0.5):")
print(f"  PR-AUC:     {val_prauc_baseline:.4f}")
print(f"  ROC-AUC:    {val_rocauc_baseline:.4f}")
print(f"  Logloss:    {val_logloss_baseline:.4f}")
print(f"  F1:         {val_f1_baseline:.4f}")
print(f"  Recall:     {val_recall_baseline:.4f}")
print(f"  Precision:  {val_precision_baseline:.4f}")

# Also train baseline WITH class weighting for comparison
print("\n--- Training Baseline (with scale_pos_weight) ---")
weighted_params = baseline_params.copy()
weighted_params['scale_pos_weight'] = scale_pos_weight_value

weighted_baseline = xgb.XGBClassifier(**weighted_params)
t0 = time.time()
weighted_baseline.fit(
    X_train_processed, y_train,
    eval_set=[(X_train_processed, y_train), (X_val_processed, y_val)],
    verbose=False
)
weighted_time = time.time() - t0

weighted_best_iter = weighted_baseline.best_iteration
val_proba_weighted = weighted_baseline.predict_proba(X_val_processed)[:, 1]
val_prauc_weighted = average_precision_score(y_val, val_proba_weighted)
val_rocauc_weighted = roc_auc_score(y_val, val_proba_weighted)
val_logloss_weighted = float(weighted_baseline.evals_result()['validation_1']['logloss'][weighted_best_iter])

val_pred_weighted = (val_proba_weighted >= 0.5).astype(int)
val_f1_weighted = f1_score(y_val, val_pred_weighted)
val_recall_weighted = recall_score(y_val, val_pred_weighted)
val_precision_weighted = precision_score(y_val, val_pred_weighted)

print(f"Best iteration: {weighted_best_iter}")
print(f"Training time: {weighted_time:.2f}s")
print(f"\nWeighted Baseline Validation Metrics (threshold=0.5):")
print(f"  PR-AUC:     {val_prauc_weighted:.4f}")
print(f"  ROC-AUC:    {val_rocauc_weighted:.4f}")
print(f"  Logloss:    {val_logloss_weighted:.4f}")
print(f"  F1:         {val_f1_weighted:.4f}")
print(f"  Recall:     {val_recall_weighted:.4f}")
print(f"  Precision:  {val_precision_weighted:.4f}")

Training class distribution: Negative=5912, Positive=1822
Calculated scale_pos_weight: 3.2448

--- Training Baseline (no class weighting) ---


Best iteration: 218
Training time: 2.58s

Baseline Validation Metrics (threshold=0.5):
  PR-AUC:     0.9456
  ROC-AUC:    0.9836
  Logloss:    0.1228
  F1:         0.8741
  Recall:     0.9364
  Precision:  0.8196

--- Training Baseline (with scale_pos_weight) ---


Best iteration: 258
Training time: 0.42s

Weighted Baseline Validation Metrics (threshold=0.5):
  PR-AUC:     0.9425
  ROC-AUC:    0.9826
  Logloss:    0.1412
  F1:         0.8768
  Recall:     0.9912
  Precision:  0.7861


## 8. Controlled Hyperparameter Experiments

We evaluate a compact set of candidate configurations, selected to explore key dimensions (tree depth, learning rate, regularization, sampling) without exhaustive grid search.

Each candidate uses early stopping and the same validation set. The class-weighting decision (with or without `scale_pos_weight`) is carried forward from whichever baseline performed better on PR-AUC.

In [7]:
# Determine if class weighting helps (based on baseline comparison)
use_class_weight = val_prauc_weighted > val_prauc_baseline
print(f"Baseline PR-AUC (unweighted): {val_prauc_baseline:.4f}")
print(f"Baseline PR-AUC (weighted):   {val_prauc_weighted:.4f}")
print(f"Using scale_pos_weight: {use_class_weight}")

# Define candidate configurations (compact, justified set)
candidate_configs = [
    # Candidate 0: Baseline winner (already trained, included for comparison)
    {'label': 'Baseline', 'max_depth': 4, 'learning_rate': 0.05, 'min_child_weight': 3,
     'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.0},
    # Candidate 1: Shallower trees, slower learning
    {'label': 'Shallow-Slow', 'max_depth': 3, 'learning_rate': 0.03, 'min_child_weight': 3,
     'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.0},
    # Candidate 2: Deeper trees, faster learning
    {'label': 'Deep-Fast', 'max_depth': 6, 'learning_rate': 0.08, 'min_child_weight': 1,
     'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.0},
    # Candidate 3: Moderate depth, more regularization
    {'label': 'Mid-HighReg', 'max_depth': 4, 'learning_rate': 0.05, 'min_child_weight': 5,
     'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 5.0, 'reg_alpha': 0.0},
    # Candidate 4: Deeper with strong regularization
    {'label': 'Deep-Reg', 'max_depth': 5, 'learning_rate': 0.05, 'min_child_weight': 3,
     'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 3.0, 'reg_alpha': 0.0},
    # Candidate 5: Full sampling, moderate depth
    {'label': 'FullSample', 'max_depth': 4, 'learning_rate': 0.05, 'min_child_weight': 3,
     'subsample': 1.0, 'colsample_bytree': 1.0, 'reg_lambda': 1.0, 'reg_alpha': 0.0},
    # Candidate 6: Shallow + high regularization (low variance)
    {'label': 'Shallow-HighReg', 'max_depth': 3, 'learning_rate': 0.05, 'min_child_weight': 5,
     'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 5.0, 'reg_alpha': 0.0},
    # Candidate 7: Deep slow learner
    {'label': 'Deep-Slow', 'max_depth': 5, 'learning_rate': 0.03, 'min_child_weight': 3,
     'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.0},
]

results = []

for i, cfg in enumerate(candidate_configs):
    label = cfg['label']
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'n_estimators': 500,
        'learning_rate': cfg['learning_rate'],
        'max_depth': cfg['max_depth'],
        'min_child_weight': cfg['min_child_weight'],
        'subsample': cfg['subsample'],
        'colsample_bytree': cfg['colsample_bytree'],
        'reg_alpha': cfg['reg_alpha'],
        'reg_lambda': cfg['reg_lambda'],
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
        'tree_method': 'hist',
        'early_stopping_rounds': 30,
    }
    if use_class_weight:
        params['scale_pos_weight'] = scale_pos_weight_value
    
    model = xgb.XGBClassifier(**params)
    t0 = time.time()
    model.fit(
        X_train_processed, y_train,
        eval_set=[(X_train_processed, y_train), (X_val_processed, y_val)],
        verbose=False
    )
    elapsed = time.time() - t0
    
    best_iter = model.best_iteration
    val_proba = model.predict_proba(X_val_processed)[:, 1]
    train_proba = model.predict_proba(X_train_processed)[:, 1]
    
    val_prauc = average_precision_score(y_val, val_proba)
    val_rocauc = roc_auc_score(y_val, val_proba)
    val_logloss = float(model.evals_result()['validation_1']['logloss'][best_iter])
    train_logloss = float(model.evals_result()['validation_0']['logloss'][best_iter])
    train_prauc = average_precision_score(y_train, train_proba)
    
    val_pred = (val_proba >= 0.5).astype(int)
    val_f1 = f1_score(y_val, val_pred)
    val_rec = recall_score(y_val, val_pred)
    val_prec = precision_score(y_val, val_pred)
    tn, fp, fn, tp = confusion_matrix(y_val, val_pred).ravel()
    val_spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    
    results.append({
        'Candidate': label,
        'max_depth': cfg['max_depth'],
        'learning_rate': cfg['learning_rate'],
        'min_child_weight': cfg['min_child_weight'],
        'subsample': cfg['subsample'],
        'colsample_bytree': cfg['colsample_bytree'],
        'reg_lambda': cfg['reg_lambda'],
        'Val_PR_AUC': round(val_prauc, 4),
        'Val_ROC_AUC': round(val_rocauc, 4),
        'Val_Logloss': round(val_logloss, 4),
        'Train_Logloss': round(train_logloss, 4),
        'Train_PR_AUC': round(train_prauc, 4),
        'Val_F1_0.5': round(val_f1, 4),
        'Val_Recall_0.5': round(val_rec, 4),
        'Val_Precision_0.5': round(val_prec, 4),
        'Val_Specificity_0.5': round(val_spec, 4),
        'Best_Iteration': best_iter,
        'Time_s': round(elapsed, 2),
    })
    print(f"  [{i}] {label:18s} | PR-AUC={val_prauc:.4f} | ROC-AUC={val_rocauc:.4f} | "
          f"F1={val_f1:.4f} | Recall={val_rec:.4f} | Iter={best_iter} | {elapsed:.1f}s")

results_df = pd.DataFrame(results)
print("\n--- All Candidate Results ---")
print(results_df.to_string(index=False))

Baseline PR-AUC (unweighted): 0.9456
Baseline PR-AUC (weighted):   0.9425
Using scale_pos_weight: False


  [0] Baseline           | PR-AUC=0.9456 | ROC-AUC=0.9836 | F1=0.8741 | Recall=0.9364 | Iter=218 | 0.3s


  [1] Shallow-Slow       | PR-AUC=0.9451 | ROC-AUC=0.9835 | F1=0.8763 | Recall=0.9474 | Iter=391 | 0.7s


  [2] Deep-Fast          | PR-AUC=0.9444 | ROC-AUC=0.9832 | F1=0.8809 | Recall=0.9408 | Iter=91 | 0.3s


  [3] Mid-HighReg        | PR-AUC=0.9455 | ROC-AUC=0.9835 | F1=0.8725 | Recall=0.9232 | Iter=273 | 0.6s


  [4] Deep-Reg           | PR-AUC=0.9454 | ROC-AUC=0.9836 | F1=0.8776 | Recall=0.9276 | Iter=229 | 0.5s


  [5] FullSample         | PR-AUC=0.9455 | ROC-AUC=0.9837 | F1=0.8761 | Recall=0.9539 | Iter=181 | 0.4s


  [6] Shallow-HighReg    | PR-AUC=0.9457 | ROC-AUC=0.9836 | F1=0.8777 | Recall=0.9364 | Iter=300 | 0.5s


  [7] Deep-Slow          | PR-AUC=0.9456 | ROC-AUC=0.9835 | F1=0.8716 | Recall=0.9452 | Iter=247 | 0.6s

--- All Candidate Results ---
      Candidate  max_depth  learning_rate  min_child_weight  subsample  colsample_bytree  reg_lambda  Val_PR_AUC  Val_ROC_AUC  Val_Logloss  Train_Logloss  Train_PR_AUC  Val_F1_0.5  Val_Recall_0.5  Val_Precision_0.5  Val_Specificity_0.5  Best_Iteration  Time_s
       Baseline          4           0.05                 3        0.8               0.8         1.0      0.9456       0.9836       0.1228         0.0994        0.9792      0.8741          0.9364             0.8196               0.9364             218    0.33
   Shallow-Slow          3           0.03                 3        0.8               0.8         1.0      0.9451       0.9835       0.1235         0.1117        0.9654      0.8763          0.9474             0.8151               0.9337             391    0.70
      Deep-Fast          6           0.08                 1        0.8               

## 9. Candidate Comparison and Model Selection

Select the winning model based primarily on **validation PR-AUC**, with recall, F1, specificity, and overfitting gap as supporting criteria.

In [8]:
# Primary selection: highest validation PR-AUC
best_idx = results_df['Val_PR_AUC'].idxmax()
best_candidate = results_df.loc[best_idx]

print(f"Selected candidate: {best_candidate['Candidate']}")
print(f"  Val PR-AUC:    {best_candidate['Val_PR_AUC']}")
print(f"  Val ROC-AUC:   {best_candidate['Val_ROC_AUC']}")
print(f"  Val F1 (0.5):  {best_candidate['Val_F1_0.5']}")
print(f"  Val Recall:    {best_candidate['Val_Recall_0.5']}")
print(f"  Best Iter:     {best_candidate['Best_Iteration']}")
print(f"  Train PR-AUC:  {best_candidate['Train_PR_AUC']}")
print(f"  Overfitting gap (Train-Val PR-AUC): {best_candidate['Train_PR_AUC'] - best_candidate['Val_PR_AUC']:.4f}")

# Rebuild selected configuration
selected_config = candidate_configs[best_idx]

# Retrain the winning model (needed to access its evals_result for learning curves)
final_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'n_estimators': 500,
    'learning_rate': selected_config['learning_rate'],
    'max_depth': selected_config['max_depth'],
    'min_child_weight': selected_config['min_child_weight'],
    'subsample': selected_config['subsample'],
    'colsample_bytree': selected_config['colsample_bytree'],
    'reg_alpha': selected_config['reg_alpha'],
    'reg_lambda': selected_config['reg_lambda'],
    'random_state': RANDOM_STATE,
    'n_jobs': -1,
    'tree_method': 'hist',
    'early_stopping_rounds': 30,
}
if use_class_weight:
    final_params['scale_pos_weight'] = scale_pos_weight_value

selected_model = xgb.XGBClassifier(**final_params)
t0 = time.time()
selected_model.fit(
    X_train_processed, y_train,
    eval_set=[(X_train_processed, y_train), (X_val_processed, y_val)],
    verbose=False
)
selected_train_time = time.time() - t0
selected_best_iter = selected_model.best_iteration

# Save training history from the selected model
evals_result = selected_model.evals_result()
training_history = pd.DataFrame({
    'iteration': range(len(evals_result['validation_0']['logloss'])),
    'train_logloss': evals_result['validation_0']['logloss'],
    'val_logloss': evals_result['validation_1']['logloss'],
})

print(f"\nSelected model retrained: best_iteration={selected_best_iter}, time={selected_train_time:.2f}s")
print(f"Training history length: {len(training_history)} iterations")

Selected candidate: Shallow-HighReg
  Val PR-AUC:    0.9457
  Val ROC-AUC:   0.9836
  Val F1 (0.5):  0.8777
  Val Recall:    0.9364
  Best Iter:     300
  Train PR-AUC:  0.9681
  Overfitting gap (Train-Val PR-AUC): 0.0224



Selected model retrained: best_iteration=300, time=0.50s
Training history length: 331 iterations


## 10. Threshold Selection on Validation Data

**Rule**: Maximize validation F1-score among thresholds with RA recall ≥ 90%.
If no threshold satisfies the constraint, fall back to the threshold that maximizes validation F1 without constraint.

In [9]:
# Generate validation probabilities
val_proba_selected = selected_model.predict_proba(X_val_processed)[:, 1]

# Sweep thresholds
thresholds = np.arange(0.10, 0.90, 0.01)
threshold_results = []

for t in thresholds:
    preds = (val_proba_selected >= t).astype(int)
    if preds.sum() == 0:
        continue
    rec = recall_score(y_val, preds)
    prec = precision_score(y_val, preds, zero_division=0)
    f1 = f1_score(y_val, preds)
    tn, fp, fn, tp = confusion_matrix(y_val, preds).ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    threshold_results.append({
        'threshold': round(t, 2),
        'recall': rec,
        'precision': prec,
        'f1': f1,
        'specificity': spec,
        'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn
    })

thresh_df = pd.DataFrame(threshold_results)

# Primary: maximize F1 among thresholds with recall >= 0.90
constrained = thresh_df[thresh_df['recall'] >= 0.90]

if len(constrained) > 0:
    best_thresh_idx = constrained['f1'].idxmax()
    selected_threshold = constrained.loc[best_thresh_idx, 'threshold']
    threshold_method = "Maximized F1 among thresholds with RA recall >= 90%"
    print(f"Constrained threshold search found {len(constrained)} thresholds with recall >= 90%.")
else:
    # Fallback: maximize F1 without constraint
    best_thresh_idx = thresh_df['f1'].idxmax()
    selected_threshold = thresh_df.loc[best_thresh_idx, 'threshold']
    threshold_method = "FALLBACK: Maximized F1 (no threshold met recall >= 90% constraint)"
    print(f"WARNING: No threshold achieved recall >= 90%. Using fallback rule.")

# Report selected threshold metrics
sel_row = thresh_df[thresh_df['threshold'] == selected_threshold].iloc[0]
print(f"\nSelected threshold: {selected_threshold}")
print(f"Method: {threshold_method}")
print(f"Validation F1:          {sel_row['f1']:.4f}")
print(f"Validation Recall:      {sel_row['recall']:.4f}")
print(f"Validation Precision:   {sel_row['precision']:.4f}")
print(f"Validation Specificity: {sel_row['specificity']:.4f}")
print(f"Validation TP={int(sel_row['tp'])}, FP={int(sel_row['fp'])}, TN={int(sel_row['tn'])}, FN={int(sel_row['fn'])}")

# Show top thresholds near the constraint boundary
print("\n--- Top thresholds by F1 (with recall >= 90%) ---")
if len(constrained) > 0:
    print(constrained.sort_values('f1', ascending=False).head(10).to_string(index=False))

print("\n✅ Threshold frozen before test evaluation.")

Constrained threshold search found 46 thresholds with recall >= 90%.

Selected threshold: 0.46
Method: Maximized F1 among thresholds with RA recall >= 90%
Validation F1:          0.8790
Validation Recall:      0.9561
Validation Precision:   0.8134
Validation Specificity: 0.9323
Validation TP=436, FP=100, TN=1378, FN=20

--- Top thresholds by F1 (with recall >= 90%) ---
 threshold   recall  precision       f1  specificity  tp  fp   tn  fn
      0.46 0.956140   0.813433 0.879032     0.932341 436 100 1378  20
      0.47 0.951754   0.815789 0.878543     0.933694 434  98 1380  22
      0.45 0.962719   0.806985 0.878000     0.928958 439 105 1373  17
      0.50 0.936404   0.825919 0.877698     0.939107 427  90 1388  29
      0.36 0.984649   0.790493 0.876953     0.919486 449 119 1359   7
      0.26 1.000000   0.780822 0.876923     0.913396 456 128 1350   0
      0.44 0.967105   0.801818 0.876740     0.926252 441 109 1369  15
      0.35 0.984649   0.789104 0.876098     0.918809 449 120 1358   

## 11. Final Held-Out Test Evaluation

All model-development decisions (hyperparameters, class weighting, threshold) are **frozen**.
The test set is evaluated exactly once for final reporting.

In [10]:
# Generate test predictions
test_proba = selected_model.predict_proba(X_test_processed)[:, 1]
test_preds = (test_proba >= selected_threshold).astype(int)

# Calculate test metrics
test_accuracy = accuracy_score(y_test, test_preds)
test_precision = precision_score(y_test, test_preds)
test_recall = recall_score(y_test, test_preds)
test_f1 = f1_score(y_test, test_preds)
test_rocauc = roc_auc_score(y_test, test_proba)
test_prauc = average_precision_score(y_test, test_proba)

tn, fp, fn, tp = confusion_matrix(y_test, test_preds).ravel()
test_specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

print("=" * 60)
print("FINAL HELD-OUT TEST EVALUATION")
print("=" * 60)
print(f"Selected threshold: {selected_threshold}")
print(f"Test records:       {len(y_test)} (RA: {int(y_test.sum())}, Non-RA: {int((y_test == 0).sum())})")
print(f"")
print(f"Accuracy:           {test_accuracy:.4f}")
print(f"Precision:          {test_precision:.4f}")
print(f"Recall/Sensitivity: {test_recall:.4f}")
print(f"Specificity:        {test_specificity:.4f}")
print(f"F1-Score:           {test_f1:.4f}")
print(f"ROC-AUC:            {test_rocauc:.4f}")
print(f"PR-AUC:             {test_prauc:.4f}")
print(f"")
print(f"Confusion Matrix:")
print(f"  True Positives  (TP): {tp}")
print(f"  False Negatives (FN): {fn}")
print(f"  True Negatives  (TN): {tn}")
print(f"  False Positives (FP): {fp}")
print("=" * 60)

FINAL HELD-OUT TEST EVALUATION
Selected threshold: 0.46
Test records:       2417 (RA: 570, Non-RA: 1847)

Accuracy:           0.9346
Precision:          0.8065
Recall/Sensitivity: 0.9509
Specificity:        0.9296
F1-Score:           0.8728
ROC-AUC:            0.9850
PR-AUC:             0.9511

Confusion Matrix:
  True Positives  (TP): 542
  False Negatives (FN): 28
  True Negatives  (TN): 1717
  False Positives (FP): 130


## 12. Visualizations

In [11]:
# Create output directory
model_dir = root / 'models' / 'xgboost'
model_dir.mkdir(parents=True, exist_ok=True)
print(f"Model directory: {model_dir}")

Model directory: D:\rheumatoid-arthritis-classification\RA_Project\models\xgboost


In [12]:
# 12a. Training and Validation Learning Curve
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(training_history['iteration'], training_history['train_logloss'],
        label='Train Logloss', color='#2196F3', linewidth=1.5)
ax.plot(training_history['iteration'], training_history['val_logloss'],
        label='Validation Logloss', color='#FF5722', linewidth=1.5)
ax.axvline(x=selected_best_iter, color='#4CAF50', linestyle='--', linewidth=1,
           label=f'Best Iteration ({selected_best_iter})')
ax.set_xlabel('Boosting Iteration', fontsize=12)
ax.set_ylabel('Log Loss', fontsize=12)
ax.set_title('XGBoost Training & Validation Learning Curve', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(model_dir / 'xgboost_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: xgboost_learning_curve.png")

Saved: xgboost_learning_curve.png


In [13]:
# 12b. Validation Candidate Comparison Plot
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

candidates = results_df['Candidate'].values
x_pos = np.arange(len(candidates))

# PR-AUC comparison
colors = ['#4CAF50' if c == best_candidate['Candidate'] else '#90CAF9' for c in candidates]
axes[0].barh(x_pos, results_df['Val_PR_AUC'], color=colors, edgecolor='#333')
axes[0].set_yticks(x_pos)
axes[0].set_yticklabels(candidates, fontsize=9)
axes[0].set_xlabel('Validation PR-AUC', fontsize=11)
axes[0].set_title('PR-AUC by Candidate', fontsize=12)
for i, v in enumerate(results_df['Val_PR_AUC']):
    axes[0].text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=8)

# ROC-AUC comparison
axes[1].barh(x_pos, results_df['Val_ROC_AUC'], color=colors, edgecolor='#333')
axes[1].set_yticks(x_pos)
axes[1].set_yticklabels(candidates, fontsize=9)
axes[1].set_xlabel('Validation ROC-AUC', fontsize=11)
axes[1].set_title('ROC-AUC by Candidate', fontsize=12)
for i, v in enumerate(results_df['Val_ROC_AUC']):
    axes[1].text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=8)

# F1 comparison
axes[2].barh(x_pos, results_df['Val_F1_0.5'], color=colors, edgecolor='#333')
axes[2].set_yticks(x_pos)
axes[2].set_yticklabels(candidates, fontsize=9)
axes[2].set_xlabel('Validation F1 (threshold=0.5)', fontsize=11)
axes[2].set_title('F1 by Candidate', fontsize=12)
for i, v in enumerate(results_df['Val_F1_0.5']):
    axes[2].text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig(model_dir / 'xgboost_validation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: xgboost_validation_comparison.png")

Saved: xgboost_validation_comparison.png


In [14]:
# 12c. Confusion Matrix (Test Set)
cm = confusion_matrix(y_test, test_preds)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
ax.figure.colorbar(im, ax=ax)
ax.set(xticks=[0, 1], yticks=[0, 1],
       xticklabels=['Non-RA (0)', 'RA (1)'],
       yticklabels=['Non-RA (0)', 'RA (1)'],
       ylabel='True Label', xlabel='Predicted Label')
ax.set_title(f'XGBoost Test Confusion Matrix (threshold={selected_threshold})', fontsize=13)

# Add counts and percentages
for i in range(2):
    for j in range(2):
        text_color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        row_total = cm[i].sum()
        pct = cm[i, j] / row_total * 100 if row_total > 0 else 0
        ax.text(j, i, f'{cm[i, j]}\n({pct:.1f}%)',
                ha='center', va='center', color=text_color, fontsize=14)

plt.tight_layout()
plt.savefig(model_dir / 'xgboost_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: xgboost_confusion_matrix.png")

Saved: xgboost_confusion_matrix.png


In [15]:
# 12d. Precision-Recall Curve (Test Set)
precision_curve, recall_curve, _ = precision_recall_curve(y_test, test_proba)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(recall_curve, precision_curve, color='#2196F3', linewidth=2,
        label=f'XGBoost PR-AUC = {test_prauc:.4f}')
ax.axhline(y=y_test.mean(), color='gray', linestyle='--', linewidth=1,
           label=f'Baseline (prevalence = {y_test.mean():.3f})')

# Mark the operating point
ax.scatter([test_recall], [test_precision], color='red', s=100, zorder=5,
           label=f'Threshold={selected_threshold} (P={test_precision:.3f}, R={test_recall:.3f})')

ax.set_xlabel('Recall (Sensitivity)', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('XGBoost Precision-Recall Curve (Test Set)', fontsize=14)
ax.legend(loc='lower left', fontsize=10)
ax.set_xlim([0.0, 1.05])
ax.set_ylim([0.0, 1.05])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(model_dir / 'xgboost_precision_recall_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: xgboost_precision_recall_curve.png")

Saved: xgboost_precision_recall_curve.png


In [16]:
# 12e. ROC Curve (Test Set)
fpr, tpr, _ = roc_curve(y_test, test_proba)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, color='#FF5722', linewidth=2,
        label=f'XGBoost ROC-AUC = {test_rocauc:.4f}')
ax.plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1,
        label='Random Classifier')

# Mark operating point
ax.scatter([1 - test_specificity], [test_recall], color='red', s=100, zorder=5,
           label=f'Threshold={selected_threshold} (FPR={1-test_specificity:.3f}, TPR={test_recall:.3f})')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate (Sensitivity)', fontsize=12)
ax.set_title('XGBoost ROC Curve (Test Set)', fontsize=14)
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(model_dir / 'xgboost_roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: xgboost_roc_curve.png")

Saved: xgboost_roc_curve.png


In [17]:
# 12f. Feature Importance Plot
# Note: Feature importance reflects model reliance / association, NOT causal influence.
importances = selected_model.feature_importances_
feat_imp_df = pd.DataFrame({
    'Feature': transformed_feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(feat_imp_df['Feature'], feat_imp_df['Importance'],
        color='#2196F3', edgecolor='#1565C0')
ax.set_xlabel('Feature Importance (Gain)', fontsize=12)
ax.set_title('XGBoost Feature Importance\n(Reflects model reliance, not causal influence)', fontsize=13)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(model_dir / 'xgboost_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: xgboost_feature_importance.png")

Saved: xgboost_feature_importance.png


## 13. Save Final Artifacts

Save all required artifacts under `models/xgboost/`.

In [18]:
model_dir = root / 'models' / 'xgboost'
model_dir.mkdir(parents=True, exist_ok=True)

# 1. Save native XGBoost model
native_model_path = model_dir / 'xgboost_model.json'
selected_model.save_model(str(native_model_path))
print(f"Saved: {native_model_path}")

# 2. Save complete inference pipeline (preprocessor + model)
# Create a sklearn Pipeline wrapping the preprocessor and XGBClassifier
inference_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', selected_model)
])
pipeline_path = model_dir / 'xgboost_pipeline.joblib'
joblib.dump(inference_pipeline, str(pipeline_path))
print(f"Saved: {pipeline_path}")

# 3. Validation metrics for the selected model
val_proba_final = selected_model.predict_proba(X_val_processed)[:, 1]
val_preds_final = (val_proba_final >= selected_threshold).astype(int)
val_tn, val_fp, val_fn, val_tp = confusion_matrix(y_val, val_preds_final).ravel()

validation_metrics_at_threshold = {
    'Threshold': selected_threshold,
    'Val_Accuracy': float(accuracy_score(y_val, val_preds_final)),
    'Val_Precision': float(precision_score(y_val, val_preds_final)),
    'Val_Recall_Sensitivity': float(recall_score(y_val, val_preds_final)),
    'Val_Specificity': float(val_tn / (val_tn + val_fp)),
    'Val_F1_Score': float(f1_score(y_val, val_preds_final)),
    'Val_ROC_AUC': float(roc_auc_score(y_val, val_proba_final)),
    'Val_PR_AUC': float(average_precision_score(y_val, val_proba_final)),
    'Val_Logloss': float(evals_result['validation_1']['logloss'][selected_best_iter]),
    'Val_TP': int(val_tp), 'Val_FP': int(val_fp),
    'Val_TN': int(val_tn), 'Val_FN': int(val_fn),
}

# 3. Save metadata
metadata = {
    'model_name': 'XGBoost Classifier',
    'random_seed': RANDOM_STATE,
    'input_features_count': 14,
    'input_features': feature_names,
    'transformed_features_count': len(transformed_feature_names),
    'transformed_feature_names': transformed_feature_names,
    'preprocessing': {
        'numeric_features': NUMERIC_FEATURES,
        'numeric_strategy': 'Passthrough (NaN preserved for XGBoost native handling)',
        'categorical_features': CATEGORICAL_FEATURES,
        'categorical_strategy': "SimpleImputer('Missing') + OneHotEncoder(handle_unknown='ignore')",
    },
    'hyperparameters': {
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'n_estimators': 500,
        'learning_rate': selected_config['learning_rate'],
        'max_depth': selected_config['max_depth'],
        'min_child_weight': selected_config['min_child_weight'],
        'subsample': selected_config['subsample'],
        'colsample_bytree': selected_config['colsample_bytree'],
        'reg_alpha': selected_config['reg_alpha'],
        'reg_lambda': selected_config['reg_lambda'],
        'tree_method': 'hist',
        'early_stopping_rounds': 30,
        'scale_pos_weight': scale_pos_weight_value if use_class_weight else None,
    },
    'training_configuration': {
        'best_iteration': int(selected_best_iter),
        'training_time_s': round(selected_train_time, 2),
        'use_class_weight': use_class_weight,
        'scale_pos_weight': round(scale_pos_weight_value, 4) if use_class_weight else None,
        'internal_train_size': int(X_train.shape[0]),
        'validation_size': int(X_val.shape[0]),
        'test_size': int(X_test.shape[0]),
    },
    'decision_threshold': {
        'selected': selected_threshold,
        'default': 0.5,
        'rationale': threshold_method,
    },
    'validation_metrics': validation_metrics_at_threshold,
    'test_metrics': {
        'Threshold': selected_threshold,
        'Accuracy': float(test_accuracy),
        'Precision': float(test_precision),
        'Recall_Sensitivity': float(test_recall),
        'Specificity': float(test_specificity),
        'F1_Score': float(test_f1),
        'ROC_AUC': float(test_rocauc),
        'PR_AUC': float(test_prauc),
        'TP': int(tp), 'FP': int(fp),
        'TN': int(tn), 'FN': int(fn),
        'Support_RA': int(y_test.sum()),
        'Support_Non_RA': int((y_test == 0).sum()),
    },
    'dependencies': {
        'python': sys.version.split()[0],
        'xgboost': xgb.__version__,
        'scikit_learn': sklearn.__version__,
        'pandas': pd.__version__,
        'numpy': np.__version__,
        'joblib': joblib.__version__,
    },
    'created_at': datetime.now(timezone.utc).isoformat(),
}

metadata_path = model_dir / 'xgboost_metadata.json'
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
print(f"Saved: {metadata_path}")

# 4. Save validation results CSV
val_results_path = model_dir / 'xgboost_validation_results.csv'
results_df.to_csv(val_results_path, index=False)
print(f"Saved: {val_results_path}")

# 5. Save training history CSV
history_path = model_dir / 'xgboost_training_history.csv'
training_history.to_csv(history_path, index=False)
print(f"Saved: {history_path}")

# 6. Save test metrics JSON
test_metrics_dict = {
    'at_selected_threshold': {
        'Threshold': selected_threshold,
        'Accuracy': float(test_accuracy),
        'Precision': float(test_precision),
        'Recall_Sensitivity': float(test_recall),
        'Specificity': float(test_specificity),
        'F1_Score': float(test_f1),
        'ROC_AUC': float(test_rocauc),
        'PR_AUC': float(test_prauc),
        'TP': int(tp), 'FP': int(fp),
        'TN': int(tn), 'FN': int(fn),
        'Support_RA': int(y_test.sum()),
        'Support_Non_RA': int((y_test == 0).sum()),
    },
    'at_default_threshold': {
        'Threshold': 0.5,
        'Accuracy': float(accuracy_score(y_test, (test_proba >= 0.5).astype(int))),
        'Precision': float(precision_score(y_test, (test_proba >= 0.5).astype(int))),
        'Recall_Sensitivity': float(recall_score(y_test, (test_proba >= 0.5).astype(int))),
        'Specificity': float(confusion_matrix(y_test, (test_proba >= 0.5).astype(int)).ravel()[0] / 
                            (confusion_matrix(y_test, (test_proba >= 0.5).astype(int)).ravel()[0] + 
                             confusion_matrix(y_test, (test_proba >= 0.5).astype(int)).ravel()[1])),
        'F1_Score': float(f1_score(y_test, (test_proba >= 0.5).astype(int))),
        'ROC_AUC': float(test_rocauc),
        'PR_AUC': float(test_prauc),
    }
}

test_metrics_path = model_dir / 'xgboost_test_metrics.json'
with open(test_metrics_path, 'w', encoding='utf-8') as f:
    json.dump(test_metrics_dict, f, indent=2)
print(f"Saved: {test_metrics_path}")

# 7. Save test predictions CSV
test_predictions_df = pd.DataFrame({
    'patient_row_id': test_row_ids,
    'y_true': y_test.values,
    'y_proba': test_proba,
    'y_pred': test_preds,
})
test_pred_path = model_dir / 'xgboost_test_predictions.csv'
test_predictions_df.to_csv(test_pred_path, index=False)
print(f"Saved: {test_pred_path}")

print("\n✅ All artifacts saved successfully.")

Saved: D:\rheumatoid-arthritis-classification\RA_Project\models\xgboost\xgboost_model.json
Saved: D:\rheumatoid-arthritis-classification\RA_Project\models\xgboost\xgboost_pipeline.joblib
Saved: D:\rheumatoid-arthritis-classification\RA_Project\models\xgboost\xgboost_metadata.json
Saved: D:\rheumatoid-arthritis-classification\RA_Project\models\xgboost\xgboost_validation_results.csv
Saved: D:\rheumatoid-arthritis-classification\RA_Project\models\xgboost\xgboost_training_history.csv
Saved: D:\rheumatoid-arthritis-classification\RA_Project\models\xgboost\xgboost_test_metrics.json
Saved: D:\rheumatoid-arthritis-classification\RA_Project\models\xgboost\xgboost_test_predictions.csv

✅ All artifacts saved successfully.


## 14. Reload and Verify Artifacts

Verify that saved artifacts can be loaded and produce consistent results without retraining.

In [19]:
print("=== Artifact Reload and Verification ===")
all_checks_passed = True

# 1. Reload native XGBoost model
reloaded_native = xgb.XGBClassifier()
reloaded_native.load_model(str(native_model_path))
print("✅ [1] Native XGBoost model reloaded.")

# 2. Reload inference pipeline
reloaded_pipeline = joblib.load(str(pipeline_path))
print("✅ [2] Inference pipeline reloaded.")

# 3. Transform a small sample of raw test features using the reloaded preprocessor
sample_raw = X_test.head(10).copy()
sample_processed = reloaded_pipeline.named_steps['preprocessor'].transform(sample_raw)
print(f"✅ [3] Sample of 10 raw test records transformed. Shape: {sample_processed.shape}")

# 4. Generate predictions with the reloaded model
sample_proba_native = reloaded_native.predict_proba(sample_processed)[:, 1]
sample_proba_pipeline = reloaded_pipeline.predict_proba(sample_raw)[:, 1]
print(f"✅ [4] Predictions generated from reloaded model and pipeline.")

# 5. Verify probabilities are finite and between 0 and 1
assert np.all(np.isfinite(sample_proba_native)), "Non-finite probabilities from native model!"
assert np.all((sample_proba_native >= 0) & (sample_proba_native <= 1)), "Probabilities out of [0,1]!"
assert np.all(np.isfinite(sample_proba_pipeline)), "Non-finite probabilities from pipeline!"
assert np.all((sample_proba_pipeline >= 0) & (sample_proba_pipeline <= 1)), "Probabilities out of [0,1]!"
print("✅ [5] All probabilities are finite and in [0, 1].")

# 6. Verify input feature schema and order
reloaded_meta = json.load(open(metadata_path, 'r', encoding='utf-8'))
assert reloaded_meta['input_features'] == feature_names, "Feature schema mismatch!"
assert reloaded_meta['input_features_count'] == 14, "Feature count mismatch!"
print("✅ [6] Input feature schema and order match the trained model.")

# 7. Verify reloaded predictions match original predictions
full_test_proba_reloaded = reloaded_pipeline.predict_proba(X_test)[:, 1]
max_diff = np.max(np.abs(full_test_proba_reloaded - test_proba))
assert max_diff < 1e-6, f"Prediction mismatch: max difference = {max_diff}"
print(f"✅ [7] Reloaded predictions match original (max diff: {max_diff:.2e}).")

# 8. Verify saved test predictions agree with saved metrics
saved_preds = pd.read_csv(test_pred_path)
saved_metrics = json.load(open(test_metrics_path, 'r'))
recomputed_acc = accuracy_score(saved_preds['y_true'], saved_preds['y_pred'])
saved_acc = saved_metrics['at_selected_threshold']['Accuracy']
assert abs(recomputed_acc - saved_acc) < 1e-6, f"Accuracy mismatch: {recomputed_acc} vs {saved_acc}"
recomputed_prauc = average_precision_score(saved_preds['y_true'], saved_preds['y_proba'])
saved_prauc = saved_metrics['at_selected_threshold']['PR_AUC']
assert abs(recomputed_prauc - saved_prauc) < 1e-6, f"PR-AUC mismatch: {recomputed_prauc} vs {saved_prauc}"
print("✅ [8] Saved predictions agree with saved metrics.")

# 9. Verify all required artifacts exist and have nonzero file sizes
required_artifacts = [
    'xgboost_model.json',
    'xgboost_pipeline.joblib',
    'xgboost_metadata.json',
    'xgboost_validation_results.csv',
    'xgboost_training_history.csv',
    'xgboost_test_metrics.json',
    'xgboost_test_predictions.csv',
    'xgboost_confusion_matrix.png',
    'xgboost_precision_recall_curve.png',
    'xgboost_roc_curve.png',
    'xgboost_feature_importance.png',
    'xgboost_validation_comparison.png',
]
for artifact_name in required_artifacts:
    artifact_path = model_dir / artifact_name
    assert artifact_path.exists(), f"Missing artifact: {artifact_name}"
    assert artifact_path.stat().st_size > 0, f"Empty artifact: {artifact_name}"
print(f"✅ [9] All {len(required_artifacts)} required artifacts exist with nonzero sizes.")

# 10. Verify Disease, Target_RA, patient_row_id are NOT input features
for col in ['Disease', 'Target_RA', 'patient_row_id']:
    assert col not in reloaded_meta['input_features'], f"Prohibited column '{col}' found in features!"
    assert col not in reloaded_meta['transformed_feature_names'], f"Prohibited column '{col}' in transformed features!"
print("✅ [10] Disease, Target_RA, patient_row_id are NOT input features.")

if all_checks_passed:
    print("\n" + "=" * 60)
    print("ALL ARTIFACT VERIFICATION CHECKS PASSED")
    print("=" * 60)

=== Artifact Reload and Verification ===


✅ [1] Native XGBoost model reloaded.
✅ [2] Inference pipeline reloaded.
✅ [3] Sample of 10 raw test records transformed. Shape: (10, 27)
✅ [4] Predictions generated from reloaded model and pipeline.
✅ [5] All probabilities are finite and in [0, 1].
✅ [6] Input feature schema and order match the trained model.
✅ [7] Reloaded predictions match original (max diff: 0.00e+00).


✅ [8] Saved predictions agree with saved metrics.
✅ [9] All 12 required artifacts exist with nonzero sizes.
✅ [10] Disease, Target_RA, patient_row_id are NOT input features.

ALL ARTIFACT VERIFICATION CHECKS PASSED


## 15. Results Summary and Limitations

In [20]:
# Compare with ANN baseline
ann_metrics_path = root / 'models' / 'ann' / 'ann_test_metrics.json'
ann_metrics = None
if ann_metrics_path.exists():
    with open(ann_metrics_path, 'r') as f:
        ann_metrics = json.load(f)
    ann_selected = ann_metrics.get('selected_threshold_0_39', {})
else:
    print("WARNING: ANN metrics file not found.")

print("=" * 70)
print("XGBOOST vs ANN — DESCRIPTIVE TEST-SET COMPARISON")
print("=" * 70)
print(f"{'Metric':<25} {'XGBoost':>12} {'ANN':>12}")
print("-" * 49)

if ann_metrics and ann_selected:
    comparisons = [
        ('Threshold', selected_threshold, ann_selected.get('Threshold', 'N/A')),
        ('Accuracy', f"{test_accuracy:.4f}", f"{ann_selected.get('Accuracy', 0):.4f}"),
        ('Precision', f"{test_precision:.4f}", f"{ann_selected.get('Precision', 0):.4f}"),
        ('Recall/Sensitivity', f"{test_recall:.4f}", f"{ann_selected.get('Recall_Sensitivity', 0):.4f}"),
        ('Specificity', f"{test_specificity:.4f}", f"{ann_selected.get('Specificity', 0):.4f}"),
        ('F1-Score', f"{test_f1:.4f}", f"{ann_selected.get('F1_Score', 0):.4f}"),
        ('ROC-AUC', f"{test_rocauc:.4f}", f"{ann_selected.get('ROC_AUC', 0):.4f}"),
        ('PR-AUC', f"{test_prauc:.4f}", f"{ann_selected.get('PR_AUC', 0):.4f}"),
    ]
    for name, xgb_val, ann_val in comparisons:
        print(f"{name:<25} {str(xgb_val):>12} {str(ann_val):>12}")

print("\n⚠️  IMPORTANT: This comparison is descriptive only.")
print("   A model must NOT be selected or retuned based on test-set comparisons.")
print("   Single test-set differences do not establish statistical superiority.")
print("   Proper model comparison requires cross-validation, confidence intervals,")
print("   or repeated trials on independent datasets.")

print("\n" + "=" * 70)
print("LIMITATIONS")
print("=" * 70)
print("1. This is a retrospective classification of recorded diagnoses, not a")
print("   clinically validated diagnostic or predictive system.")
print("2. The 90% sensitivity target is an experimental project criterion,")
print("   not a clinically validated threshold.")
print("3. Feature importance reflects model reliance/association, not causal influence.")
print("4. The dataset may not represent the full clinical diversity of RA presentation.")
print("5. Single train/test split results have inherent variance; confidence")
print("   intervals are not reported here.")
print("6. Missing data patterns may be informative (differential test ordering)")
print("   and the model may learn from missingness patterns.")

XGBOOST vs ANN — DESCRIPTIVE TEST-SET COMPARISON
Metric                         XGBoost          ANN
-------------------------------------------------
Threshold                         0.46         0.39
Accuracy                        0.9346       0.9164
Precision                       0.8065       0.7730
Recall/Sensitivity              0.9509       0.9140
Specificity                     0.9296       0.9172
F1-Score                        0.8728       0.8376
ROC-AUC                         0.9850       0.9768
PR-AUC                          0.9511       0.9252

⚠️  IMPORTANT: This comparison is descriptive only.
   A model must NOT be selected or retuned based on test-set comparisons.
   Single test-set differences do not establish statistical superiority.
   Proper model comparison requires cross-validation, confidence intervals,
   or repeated trials on independent datasets.

LIMITATIONS
1. This is a retrospective classification of recorded diagnoses, not a
   clinically validated di